In [6]:
from importlib.metadata import version

print(version("torch"))
print(version("tiktoken"))

2.13.0
0.13.0


In [10]:
import os
import urllib.request

if not os.path.exists('the-verdict.txt'):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = 'the-verdict.txt'
    #第二个参数接收文件路径和文件名
        #仅有文件保存在当前目录
        #mac开头的/是绝对路径
        #./和../等才是相对路径
    urllib.request.urlretrieve(url,file_path)

In [14]:
#r是只读模式
#文本编码格式为utf8 写入时候也是utf8
#as f将这里的东西赋值给f,结束时候收回f
with open('the-verdict.txt','r',encoding = 'utf-8') as f:
    raw_text = f.read()
print('Total number of character:',len(raw_text))
#止于98,提取到第98个索引(包括)
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [18]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [20]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [21]:
#strip去除空白字符
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [28]:
text = 'Hello, world. Is this-- a test?'

#字符串里面\' '前的\表示这是一个普通的引号,防止提前结束字符串
result = re.split(r'([,.:;?_!"()\']|--|\s)',text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [29]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [30]:
print(len(preprocessed))

4690


In [31]:
#set是集合的意思,集合不能有重复的元素,所以变为集合自动去重
#排序是提取集合的元素排序然后返回列表
    #不同(互不兼容的类型)不能放在一起排序,根据类型及其对应规则排序
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [32]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [33]:
#.items()这个也是可以被enumerate迭代的,返回的既不是列表也不是元组,每次迭代返回(token,integer)元组
for i,item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [48]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        #正则表达式中的 \s 匹配所有空白字符
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)

        preprocessed = [
            #strip返回字符串去掉周围空白字符
                #如果是\s这样的空白字符,.strip()直接去除所有的空白字符返回''(空字符串
                #if遇到''空字符串返回False,以此清空所有\s
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        #以空格链接所有的str字符
        text = " ".join([self.int_to_str[i] for i in ids])
        #\s匹配空格, + 表示和其他字符
        #第二个\1表示保留括号里买二内容,\s是空白符合,没有序号,第一个括号表示\1
        text = re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text

In [49]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [50]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [51]:
#在python打印时候,防止单引号与包起来的单引号重复,将句子里的单引号前加上\
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [53]:
tokenizer = SimpleTokenizerV1(vocab)
text = "Hello, do you like tea. Is this-- a test?"
tokenizer.encode(text)

KeyError: 'Hello'

In [56]:
all_token = sorted(list(set(preprocessed)))
#extend将输入的列表拆开,逐一添加到最后(多个项)
    #append是一起加入到最后的一项
all_token.extend(['<|endoftext|>','<|unk|>'])

vocab = {token:integer for integer,token in enumerate(all_token)}

In [57]:
len(vocab.items())

1132

In [59]:
for i,item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [61]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        #[ A if 条件 else B  for item in 列表 ]
        #这里的A if 条件 else B算是一个值(python视为的),所以写在for左边 和之前的if在for后不一样
            #之前的if没有else,不是选一个值
        preprocessed = [
            #字典的if默认值配对识别键
            item if item in self.str_to_int
            else '<|unk|>' for item in preprocessed
        ]
        
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [74]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

#注意两边需要空格,不然和其他词连着被认为是一个词
text = ' <|endoftext|> '.join((text1,text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [75]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [76]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

In [77]:
import importlib
import tiktoken

In [79]:
print('tiktoken version:',importlib.metadata.version('tiktoken'))

tiktoken version: 0.13.0


In [82]:
tokenizer = tiktoken.get_encoding('gpt2')

In [86]:
#小括号里的字符串换行没分割,还是会自动拼接为一个字符串
#结果还是一个字符串,元组的核心是()下的逗号,这里没有逗号,还是字符串
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

#allowed_special是配置允许的特殊符号,给予词元id,防止注入攻击
integers = tokenizer.encode(text,allowed_special = {'<|endoftext|>'})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [87]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [88]:
test_text = 'Akwirw ier'

In [89]:
tokenizer.encode(test_text)

[33901, 86, 343, 86, 220, 959]

In [90]:
tokenizer.decode(tokenizer.encode(test_text))

'Akwirw ier'

In [93]:
#open是python内置函数,建立文件通道对象,这里还没有读取文件,这个对象的.read()才是读取内部数据
with open('the-verdict.txt','r',encoding = 'utf-8') as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [95]:
#跳过标题,完整的句子效果更好
enc_sample = enc_text[50:]

In [97]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f'x: {x}')
print(f'y:      {y}')

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [99]:
for i in range(1,context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context,'---->',desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [101]:
for i in range(1,context_size + 1):
    context = enc_sample[:i]
    #这里直接取出来的是元素,所以后面的decode输入必须是列表形式,加了[]
    desired = enc_sample[i]

    print(tokenizer.decode(context),'---->',tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [102]:
import torch
print('pytorch version:',torch.__version__)

pytorch version: 2.13.0


In [109]:
from torch.utils.data import Dataset,DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt,allowed_special = {'<|endoftext|>'})

        for i in range(0,len(token_ids) - max_length,stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1:i+max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]

In [110]:
def create_dataloader_v1(txt,batch_size = 4,max_length = 256,
                         stride = 128,shuffle = True,drop_last = True,
                         num_workers = 0):
    tokenizer = tiktoken.get_encoding('gpt2')

    dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)

    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        #是否打乱
        shuffle = shuffle,
        #如果为true,批次大小小于指定的二batchsize,删除最后一批
        drop_last = drop_last,
        #cpu进程数量
        num_workers = num_workers
    )

    return dataloader

In [111]:
with open('the-verdict.txt','r',encoding = 'utf-8') as f:
    raw_text = f.read()

In [112]:
dataloader = create_dataloader_v1(
    raw_text,batch_size = 1,max_length = 4,stride = 1,shuffle = False
)

#dataloader是一个可迭代对象,这里用iter变化成一个python迭代器 避免了用for,可以手动控制迭代节奏
    #用next输入迭代器,取出下一个数据
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [113]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [114]:
dataloader = create_dataloader_v1(
    raw_text,batch_size = 1,max_length = 2,stride = 2,shuffle = False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]
[tensor([[2885, 1464]]), tensor([[1464, 1807]])]


In [115]:
dataloader = create_dataloader_v1(
    raw_text,batch_size = 1,max_length = 8,stride = 2,shuffle = False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]
[tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]


In [118]:
dataloader = create_dataloader_v1(raw_text,batch_size = 8,max_length = 4,
                                  stride = 4,shuffle = False)

data_iter = iter(dataloader)
inputs,targets = next(data_iter)
print('Inputs:\n',inputs)
print('\nTargets:\n',targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [119]:
input_ids = torch.tensor([2,3,5,1])

In [120]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [121]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [122]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [123]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [125]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [126]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text,batch_size = 8,max_length = max_length,
    stride = max_length,shuffle = False
)
data_iter = iter(dataloader)
inputs,targets = next(data_iter)

In [127]:
print('Token IDs:\n',inputs)
print('\nInputs shape:\n',inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [130]:
#支出多维度张量输入,最里面的对应词元id,提取词嵌入
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [132]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length,output_dim)

In [133]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [136]:
#仅表示位置偏置,每一批都是一样的大小,所以每一批的位置偏置也一样即可
#这里用了广播
#这个生成的位置偏置也是一个会进行学习的参数
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
